In [ ]:
import pandas as pd
import numpy as np

In [ ]:
movie_data = pd.read_csv("movies.csv")
rating_data = pd.read_csv("ratings.csv")
links_data = pd.read_csv("links.csv")
tag_data = pd.read_csv("tags.csv")

Movie_A = 85788 # insidios (2010)

Här tar vi ut alla användare baserat på 2 regler

regel 1: måste gilla film A                                 
regel 2: måste gett film A minst 4 sjärnor

In [ ]:
liking_Movie_A = rating_data[
    (rating_data["movieId"] == Movie_A) & 
    (rating_data["rating"] >= 4.0) #minst 4 i betyg
            ].sort_values("userId")

users_A = liking_Movie_A["userId"].unique()

samlar alla filmer i grupper baserat hur många som gillat filmerna

In [ ]:
rekommend = rating_data[
    (rating_data["userId"].isin(users_A))&
    (rating_data["movieId"] != Movie_A)&
    (rating_data["rating"] >= 4)
            ].groupby("movieId").size()

rating_count = rating_data.groupby("movieId").size()

Här använder vi oss av 2 kalkulationer för att förhindra att äldre populära filmer alltid rekomenderas högst upp

In [ ]:
rekommend = rekommend / np.sqrt(rating_count)
rekommend = rekommend / len(users_A)

In [ ]:
top30 = (rekommend.sort_values(ascending=False).head(30))
top30 = top30.reset_index()

In [ ]:
top30.columns = ["movieId", "score"]
top30_score = top30.merge(movie_data, on="movieId")
top30_score.head(10)

,movieId,score,title,genres
0,103688,0.008632,"Conjuring, The (2013)",Horror|Thriller
1,104908,0.008283,Insidious: Chapter 2 (2013),Horror|Thriller
2,97188,0.007982,Sinister (2012),Horror|Thriller
3,159858,0.006391,The Conjuring 2 (2016),Horror
4,122884,0.005194,Insidious: Chapter 3 (2015),Fantasy|Horror|Thriller
5,110591,0.004769,Oculus (2013),Horror
6,70159,0.004625,Orphan (2009),Drama|Horror|Mystery|Thriller
7,57274,0.004459,[REC] (2007),Drama|Horror|Thriller
8,121231,0.004375,It Follows (2014),Horror
9,93840,0.004205,"Cabin in the Woods, The (2012)",Comedy|Horror|Sci-Fi|Thriller


valt att ha kvar allt relaterat till year då jag tänkt försöka använda det i programet i framtiden men just nu gav det inga bra resultat

In [ ]:
#movie_data["year"] = movie_data["title"].str.extract(r"\((\d{4})\)").astype(float)

#year_a = movie_data.loc[movie_data["movieId"] == Movie_A, "year"].iloc[0]

# diffrens_in_years = abs(top30_merged["year"] - year_a)
# year_diff = diffrens_in_years.reset_index()

# for i in range(len(year_diffrens)):
#     bonus = 0
#     if int(year_diffrens["year"].loc[i]) <= 2:
#         bonus = 0.1
#         top15_year.loc[i, "score"] += bonus
#     elif int(year_diffrens["year"].loc[i]) <= 5:
#         bonus = 0.06
#         top15_year.loc[i, "score"] += bonus
#     elif int(year_diffrens["year"].loc[i]) <= 8:
#         bonus = 0.03
#         top15_year.loc[i, "score"] += bonus
#     else:
#         top15_year.loc[i, "score"]

In [ ]:
top30_merge = top30_score.merge(movie_data, on=["movieId", "title", "genres"])
top30_merged = top30_merge.merge(links_data[["movieId", "imdbId"]], how="left")
top30_merged

,movieId,score,title,genres,imdbId
0,103688,0.008632,"Conjuring, The (2013)",Horror|Thriller,1457767
1,104908,0.008283,Insidious: Chapter 2 (2013),Horror|Thriller,2226417
2,97188,0.007982,Sinister (2012),Horror|Thriller,1922777
3,159858,0.006391,The Conjuring 2 (2016),Horror,3065204
4,122884,0.005194,Insidious: Chapter 3 (2015),Fantasy|Horror|Thriller,3195644
5,110591,0.004769,Oculus (2013),Horror,2388715
6,70159,0.004625,Orphan (2009),Drama|Horror|Mystery|Thriller,1148204
7,57274,0.004459,[REC] (2007),Drama|Horror|Thriller,1038988
8,121231,0.004375,It Follows (2014),Horror,3235888
9,93840,0.004205,"Cabin in the Woods, The (2012)",Comedy|Horror|Sci-Fi|Thriller,1259521


In [ ]:
top = top30_merged.sort_values("score", ascending=False)
top.head(10)

,movieId,score,title,genres,imdbId
0,103688,0.008632,"Conjuring, The (2013)",Horror|Thriller,1457767
1,104908,0.008283,Insidious: Chapter 2 (2013),Horror|Thriller,2226417
2,97188,0.007982,Sinister (2012),Horror|Thriller,1922777
3,159858,0.006391,The Conjuring 2 (2016),Horror,3065204
4,122884,0.005194,Insidious: Chapter 3 (2015),Fantasy|Horror|Thriller,3195644
5,110591,0.004769,Oculus (2013),Horror,2388715
6,70159,0.004625,Orphan (2009),Drama|Horror|Mystery|Thriller,1148204
7,57274,0.004459,[REC] (2007),Drama|Horror|Thriller,1038988
8,121231,0.004375,It Follows (2014),Horror,3235888
9,93840,0.004205,"Cabin in the Woods, The (2012)",Comedy|Horror|Sci-Fi|Thriller,1259521


In [ ]:
movie_a_genrer = movie_data[movie_data["movieId"] == Movie_A]["genres"].iloc[0].split("|")
movie_a_genrer

['Fantasy', 'Horror', 'Thriller']

In [ ]:
for i, genres in enumerate(top["genres"]):
    
    genre_list = genres.split("|")
    matches = 0

    for g in genre_list:
        if g in movie_a_genrer:
            matches += 1

    if matches > 0:
        similarity = matches / (len(movie_a_genrer) + 2)
        top.loc[i, "score"] += similarity

In [ ]:
resultat = top.sort_values("score",ascending=False).head(5)
resultat

,movieId,score,title,genres,imdbId
4,122884,0.605194,Insidious: Chapter 3 (2015),Fantasy|Horror|Thriller,3195644
0,103688,0.408632,"Conjuring, The (2013)",Horror|Thriller,1457767
1,104908,0.408283,Insidious: Chapter 2 (2013),Horror|Thriller,2226417
2,97188,0.407982,Sinister (2012),Horror|Thriller,1922777
6,70159,0.404625,Orphan (2009),Drama|Horror|Mystery|Thriller,1148204
